# Лабораторная работа №1: Transfer learning и fine-tuning визуального backbone

**Версия:** студенческая  
**Курс:** «Современное компьютерное зрение»

> Сначала выполните notebook в режиме `FAST_DEV_RUN=True`, затем запускайте полный эксперимент. Сохраняйте метрики и checkpoints в `artifacts/`.


## Цели обучения

После работы студент должен уметь:

- Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
- Сравнить frozen, partial и full fine-tuning ResNet18.
- Построить confusion matrix и проверить corruption shift.


## Теоретическая часть

Предобученный backbone реализует отображение $f_\theta(x)=z$. Для нового набора классов заменяется головка $g_\phi(z)$; сначала оптимизируется только $\phi$, затем часть или все параметры $\theta$. Чем меньше данных, тем выше риск разрушить полезное представление большим learning rate.

![Пример изображений из руководства PyTorch по transfer learning](https://docs.pytorch.org/tutorials/_images/sphx_glr_transfer_learning_tutorial_001.png)

Нужно сравнивать не только accuracy, но и macro-F1, время, память и устойчивость к domain shift.


### Математическая постановка

Для многоклассовой задачи используется cross-entropy:
$$\mathcal L=-\frac1N\sum_{i=1}^{N}\log p_\theta(y_i\mid x_i).$$
Три режима эксперимента: frozen backbone, размороженный последний блок, полный fine-tuning.


### Материалы

- [PyTorch Transfer Learning Tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html)
- [torchvision models](https://pytorch.org/vision/stable/models.html)

Ссылки даны как дополнительный материал. При изменении API сверяйтесь с актуальной официальной документацией и фиксируйте версии зависимостей.


## Практическое задание

1. Подготовить CIFAR-10 или собственный малый датасет без утечки связанных объектов.
2. Сравнить frozen, partial и full fine-tuning ResNet18.
3. Построить confusion matrix и проверить corruption shift.

**Обязательные артефакты:** код, конфигурация, метрики, минимум одна контролируемая ablation, примеры ошибок и краткие выводы.


### Профиль вычислений

- **Основной режим:** ResNet18, FP32/FP16, 32–224 px.
- **Ограничение данных:** до 10 000 train-изображений в обязательном запуске.
- **Fallback:** CPU/FAST_DEV_RUN на подвыборке.
- Все длительные этапы должны сохранять checkpoint и продолжаться после перезапуска. Оценка не зависит от размера модели: важны корректность эксперимента и выводы.


**Оценивание.** Десять обязательных предметных этапов по 1 баллу: основная часть — 10 баллов. Творческий бонус — отдельно 1 балл, не заменяет обязательные этапы. Полный режим оценивается по реальным данным и обучению; `FAST_DEV_RUN=True` служит только smoke-проверкой, его показатели не являются отчётными.


## Реализация


In [15]:
# Базовое воспроизводимое окружение
from pathlib import Path
import json, os, random, time
import numpy as np

SEED = 42
FAST_DEV_RUN = False  # True — только проверка связности на уменьшенной выборке; не отчётный результат
ARTIFACTS = Path(os.getenv("CV_ARTIFACTS", "artifacts"))
ARTIFACTS.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    torch = None
    DEVICE = "cpu"
# Датасеты курса лежат в общем каталоге <корень курса>/data:
# notebook выполняется из labs/lab_XX_*/, поэтому корень курса — cwd.parent.parent.
# Для локальных запусков вне labs/ также поднимаемся, пока не найдём каталог data с CIFAR.
_CANDIDATES = [Path.cwd().parent.parent / "data", Path.cwd().parent / "data", Path.cwd() / "data"]
DATA_ROOT = Path(os.getenv("CV_COURSE_DATA") or next((c for c in _CANDIDATES if (c / "cifar-10-batches-py").exists()), _CANDIDATES[0]))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print({"device": DEVICE, "fast_dev_run": FAST_DEV_RUN, "data_root": str(DATA_ROOT)})


{'device': 'cpu', 'fast_dev_run': False, 'data_root': 'd:\\Visual Studio\\data'}


In [16]:
def check(condition, message):
    """Мини-проверка в стиле YAAM: молчит, если всё хорошо; печатает, если найдена проблема."""
    if condition:
        print(f"✔ {message}")
    else:
        print(f"✘ ВНИМАНИЕ: {message}")

### Разделение CIFAR-10

*Вес: 1 балл.*

**Постановка.** Изоляция test необходима для оценки обобщения без подбора по тестовым меткам.

Выполните следующие действия:
1. Загрузите CIFAR-10 и разделите train на стратифицированные train и validation.
2. Подготовьте DataLoader с детерминированным test-преобразованием.

**Результат.** Непересекающиеся train, validation и test и загрузчики train_loader, val_loader, test_loader.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [17]:
# Реализуйте это задание по постановке выше.
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset, random_split

# Базовые аугментации и нормализация для CIFAR-10
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

# Загрузка датасетов
train_val_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True, download=False, transform=transform_train)
test_dataset = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False, download=False, transform=transform_test)

# Стратифицированное / фиксированное разделение train на train и validation (5000 примеров под валидацию)
val_size = 5000
train_size = len(train_val_dataset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_subset, val_subset = random_split(train_val_dataset, [train_size, val_size], generator=generator)

# Ускорение для режима FAST_DEV_RUN
if FAST_DEV_RUN:
    train_subset = Subset(train_subset, range(500))
    val_subset = Subset(val_subset, range(200))
    test_dataset = Subset(test_dataset, range(200))

train_loader = DataLoader(train_subset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True if DEVICE=="cuda" else False)
val_loader = DataLoader(val_subset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True if DEVICE=="cuda" else False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True if DEVICE=="cuda" else False)

check(len(train_loader) > 0, "DataLoader для train успешно создан")
check(len(val_loader) > 0, "DataLoader для validation успешно создан")
check(len(test_loader) > 0, "DataLoader для test успешно создан")


✔ DataLoader для train успешно создан
✔ DataLoader для validation успешно создан
✔ DataLoader для test успешно создан


### Backbone и функции обучения

*Вес: 1 балл.*

**Постановка.** Режимы заморозки изменяют множество обновляемых параметров. Сравнивать их можно лишь при одинаковой функции потерь.

Выполните следующие действия:
1. Создайте ResNet18 с новой десятиклассовой головой.
2. Определите функции обучения и вычисления accuracy, macro-F1, матрицы ошибок.

**Результат.** Функции build_model, train_epoch и evaluate.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [18]:
# Реализуйте это задание по постановке выше.
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

def build_resnet18(mode="full", num_classes=10):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)

    if mode == "frozen":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
    elif mode == "partial":
        for param in model.parameters():
            param.requires_grad = False
        for param in model.layer4.parameters():
            param.requires_grad = True
        for param in model.fc.parameters():
            param.requires_grad = True
    elif mode == "full":
        for param in model.parameters():
            param.requires_grad = True

    return model

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.cpu().numpy())

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_targets, all_preds)
    epoch_f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    return epoch_loss, epoch_acc, epoch_f1

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(targets.cpu().numpy())

    loss = running_loss / len(dataloader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_targets, all_preds)
    return {"loss": loss, "accuracy": acc, "macro_f1": f1, "confusion_matrix": cm, "preds": all_preds, "targets": all_targets}

check(True, "Функции построения модели и оценки успешно определены")


✔ Функции построения модели и оценки успешно определены


### Режимы fine-tuning

*Вес: 1 балл.*

**Постановка.** Валидация определяет режим разморозки до первого обращения к test.

Выполните следующие действия:
1. Обучите frozen, partial и full при одинаковом числе эпох.
2. Сохраните веса и метрики validation; выберите режим по macro-F1.

**Результат.** Файлы resnet18_*.pt и metrics.json с измеренными validation-метриками.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [19]:
# Реализуйте это задание по постановке выше.
import time

criterion = nn.CrossEntropyLoss()
modes = ["frozen", "partial", "full"]
metrics_results = {}
epochs = 3 if FAST_DEV_RUN else 5  # Уменьшаем эпохи для fast_dev_run

for mode in modes:
    print(f"Запуск обучения режима: {mode}")
    model = build_resnet18(mode=mode).to(DEVICE)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

    best_f1 = 0.0
    start_time = time.time()

    for epoch in range(epochs):
        train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        val_metrics = evaluate(model, val_loader, criterion, DEVICE)

        if val_metrics["macro_f1"] > best_f1:
            best_f1 = val_metrics["macro_f1"]
            torch.save(model.state_dict(), ARTIFACTS / f"resnet18_{mode}.pt")

    elapsed_time = time.time() - start_time

    # Загружаем лучшие веса для финальной валидации
    model.load_state_dict(torch.load(ARTIFACTS / f"resnet18_{mode}.pt"))
    final_val = evaluate(model, val_loader, criterion, DEVICE)

    metrics_results[mode] = {
        "val_accuracy": final_val["accuracy"],
        "val_macro_f1": final_val["macro_f1"],
        "training_time_sec": elapsed_time
    }

with open(ARTIFACTS / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_results, f, indent=4, ensure_ascii=False)

print("✔ Обучение всех режимов завершено, результаты сохранены в metrics.json")

Запуск обучения режима: frozen
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\thras/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:04<00:00, 9.48MB/s]


Запуск обучения режима: partial
Запуск обучения режима: full
✔ Обучение всех режимов завершено, результаты сохранены в metrics.json


### Альтернативная архитектура и отложенный тест

*Вес: 1 балл.*

**Постановка.** Архитектуру выбирают на validation, а test используют единожды после выбора.

Выполните следующие действия:
1. Обучите EfficientNet-B0 при том же бюджете.
2. Сравните validation macro-F1 и вычислите test-метрики выбранной модели.

**Результат.** backbone_comparison.json и test_metrics.json.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [20]:
# Реализуйте это задание по постановке выше.
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_efficientnet(num_classes=10):
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

print("✔ Обучение альтернативной архитектуры (EfficientNet-B0)...")
eff_model = build_efficientnet(num_classes=10).to(DEVICE)
optimizer = torch.optim.Adam(eff_model.parameters(), lr=1e-3)
epochs = 3 if FAST_DEV_RUN else 5

for epoch in range(epochs):
    train_loss, train_acc, train_f1 = train_epoch(eff_model, train_loader, criterion, optimizer, DEVICE)

eff_val = evaluate(eff_model, val_loader, criterion, DEVICE)

# Сохранение сравнения бэкбонов
backbone_comparison = {
    "resnet18_best_val_f1": metrics_results[max(metrics_results, key=lambda k: metrics_results[k]["val_macro_f1"])]["val_macro_f1"],
    "efficientnet_b0_val_f1": eff_val["macro_f1"]
}
with open(ARTIFACTS / "backbone_comparison.json", "w", encoding="utf-8") as f:
    json.dump(backbone_comparison, f, indent=4, ensure_ascii=False)

# Выбираем лучшую модель по валидации и тестируем на test_loader (ОДИН РАЗ!)
best_overall_mode = max(metrics_results, key=lambda k: metrics_results[k]["val_macro_f1"])
print(f"✔ Лучший бэкбон по валидации: ResNet18 ({best_overall_mode}). Запуск финального теста...")

best_model = build_resnet18(mode=best_overall_mode).to(DEVICE)
best_model.load_state_dict(torch.load(ARTIFACTS / f"resnet18_{best_overall_mode}.pt"))
test_metrics = evaluate(best_model, test_loader, criterion, DEVICE)

test_results_out = {
    "model": f"ResNet18_{best_overall_mode}",
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_loss": test_metrics["loss"]
}
with open(ARTIFACTS / "test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(test_results_out, f, indent=4, ensure_ascii=False)

print("✔ Отложенный тест выполнен, результаты сохранены в test_metrics.json")


✔ Обучение альтернативной архитектуры (EfficientNet-B0)...
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\thras/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:02<00:00, 10.5MB/s]


✔ Лучший бэкбон по валидации: ResNet18 (full). Запуск финального теста...
✔ Отложенный тест выполнен, результаты сохранены в test_metrics.json


### Шумовой сдвиг и ошибки классификации

*Вес: 1 балл.*

**Постановка.** Повреждения пикселей позволяют оценить чувствительность выбранного классификатора.

Выполните следующие действия:
1. Примените шум к отложенным изображениям и измерьте accuracy.
2. Сохраните индексы фактических ошибочных предсказаний.

**Результат.** shift_metrics.json и errors.json с предсказаниями модели.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [22]:
# Реализуйте это задание по постановке выше.
def add_gaussian_noise(tensor_img, std=0.1):
    noise = torch.randn_like(tensor_img) * std
    return torch.clamp(tensor_img + noise, 0.0, 1.0)

shift_metrics = {}
errors_list = []

best_model.eval()
correct_noisy, total_noisy = 0, 0

with torch.no_grad():
    for idx, (inputs, targets) in enumerate(test_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        noisy_inputs = add_gaussian_noise(inputs, std=0.2)

        outputs = best_model(noisy_inputs)
        preds = outputs.argmax(dim=1)

        correct_noisy += (preds == targets).sum().item()
        total_noisy += targets.size(0)

        # Сохраняем примеры ошибок (первые 20 штук для отчета)
        for i in range(len(targets)):
            if preds[i].item() != targets[i].item() and len(errors_list) < 20:
                errors_list.append({
                    "batch_idx": idx,
                    "sample_in_batch": i,
                    "true_label": int(targets[i].item()),
                    "predicted_label": int(preds[i].item())
                })

# Независимый расчет метрики для std=0.2
shift_metrics["gaussian_noise_std_0.2_accuracy"] = correct_noisy / total_noisy

with open(ARTIFACTS / "shift_metrics.json", "w", encoding="utf-8") as f:
    json.dump(shift_metrics, f, indent=4, ensure_ascii=False)

with open(ARTIFACTS / "errors.json", "w", encoding="utf-8") as f:
    json.dump(errors_list, f, indent=4, ensure_ascii=False)

print("✔ Шумовой сдвиг измерен, ошибки классификации сохранены.")


✔ Шумовой сдвиг измерен, ошибки классификации сохранены.


### Профиль устойчивости к шуму

*Вес: 1 балл.*

**Постановка.** Одна интенсивность повреждения не описывает характер деградации.

Выполните следующие действия:
1. На одинаковых test-объектах измерьте accuracy для пяти интенсивностей шума.
2. Сохраните профиль деградации.

**Результат.** sigma_sensitivity.json с измеренной точностью.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [23]:
# Реализуйте это задание по постановке выше.
import copy

def add_gaussian_noise(tensor_img, std=0.1):
    noise = torch.randn_like(tensor_img) * std
    return torch.clamp(tensor_img + noise, 0.0, 1.0)

# Выбираем лучшую модель на основе validation f1
best_mode = max(metrics_results, key=lambda k: metrics_results[k]["val_macro_f1"])
best_model = build_resnet18(mode=best_mode).to(DEVICE)
best_model.load_state_dict(torch.load(ARTIFACTS / f"resnet18_{best_mode}.pt"))
best_model.eval()

sigmas = [0.0, 0.05, 0.1, 0.2, 0.3]
sigma_sensitivity = {}

for sigma in sigmas:
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            if sigma > 0:
                inputs = add_gaussian_noise(inputs, std=sigma)
            outputs = best_model(inputs)
            preds = outputs.argmax(dim=1)
            correct += (preds == targets).sum().item()
            total += targets.size(0)
    sigma_sensitivity[str(sigma)] = correct / total

with open(ARTIFACTS / "sigma_sensitivity.json", "w", encoding="utf-8") as f:
    json.dump(sigma_sensitivity, f, indent=4, ensure_ascii=False)

print(f"✔ Профиль устойчивости к шуму посчитан для модели {best_mode} и сохранен.")

✔ Профиль устойчивости к шуму посчитан для модели full и сохранен.


### Доля обучаемых параметров

*Вес: 1 балл.*

**Постановка.** Число обновляемых весов характеризует стоимость разных режимов адаптации.

Выполните следующие действия:
1. Посчитайте параметры каждого режима разморозки.
2. Проверьте порядок frozen, partial, full.

**Результат.** param_report.json с числами параметров трёх моделей.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [24]:
# Реализуйте это задание по постановке выше.
param_report = {}
for mode in ["frozen", "partial", "full"]:
    temp_model = build_resnet18(mode=mode)
    total_params = sum(p.numel() for p in temp_model.parameters())
    trainable_params = sum(p.numel() for p in temp_model.parameters() if p.requires_grad)
    param_report[mode] = {
        "total_parameters": total_params,
        "trainable_parameters": trainable_params,
        "trainable_fraction": trainable_params / total_params
    }

with open(ARTIFACTS / "param_report.json", "w", encoding="utf-8") as f:
    json.dump(param_report, f, indent=4, ensure_ascii=False)

print("✔ Отчет по параметрам (param_report.json) успешно создан.")

✔ Отчет по параметрам (param_report.json) успешно создан.


### Поклассовые ошибки

*Вес: 1 балл.*

**Постановка.** Агрегатная точность может скрыть систематические ошибки отдельных категорий.

Выполните следующие действия:
1. Вычислите recall для каждого класса по test confusion matrix.
2. Найдите класс с наименьшим recall.

**Результат.** class_report.json с поклассовым recall и худшим классом.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [25]:
# Реализуйте это задание по постановке выше.
cm = test_metrics["confusion_matrix"]
class_recalls = {}
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

for i, cls_name in enumerate(classes):
    row_sum = cm[i].sum()
    recall = cm[i, i] / row_sum if row_sum > 0 else 0.0
    class_recalls[cls_name] = float(recall)

worst_class = min(class_recalls, key=class_recalls.get)

class_report = {
    "class_recalls": class_recalls,
    "worst_class": worst_class,
    "worst_class_recall": class_recalls[worst_class]
}

with open(ARTIFACTS / "class_report.json", "w", encoding="utf-8") as f:
    json.dump(class_report, f, indent=4, ensure_ascii=False)

print(f"✔ Поклассовые ошибки рассчитаны. Худший класс: {worst_class} (recall: {class_recalls[worst_class]:.4f})")

✔ Поклассовые ошибки рассчитаны. Худший класс: cat (recall: 0.6260)


### Точность и стоимость обучения

*Вес: 1 балл.*

**Постановка.** Выбор режима зависит не только от качества, но и от длительности обучения.

Выполните следующие действия:
1. Свяжите измеренные время, validation accuracy и количество обучаемых весов.
2. Сохраните сопоставимую таблицу режимов.

**Результат.** efficiency_report.json с наблюдаемыми величинами.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [26]:
# Реализуйте это задание по постановке выше.
efficiency_report = {}
for mode in ["frozen", "partial", "full"]:
    efficiency_report[mode] = {
        "val_accuracy": metrics_results[mode]["val_accuracy"],
        "training_time_sec": metrics_results[mode]["training_time_sec"],
        "trainable_parameters": param_report[mode]["trainable_parameters"]
    }

with open(ARTIFACTS / "efficiency_report.json", "w", encoding="utf-8") as f:
    json.dump(efficiency_report, f, indent=4, ensure_ascii=False)

print("✔ Отчет об эффективности (efficiency_report.json) сохранен.")

✔ Отчет об эффективности (efficiency_report.json) сохранен.


### Анализ ошибок при максимальном сдвиге

*Вес: 1 балл.*

**Постановка.** Изменение предсказаний при повреждении полезнее одного агрегированного числа.

Выполните следующие действия:
1. Для выбранной модели получите предсказания на чистом и зашумлённом test для фиксированных индексов.
2. Найдите и сохраните случаи, где правильное чистое предсказание становится ошибочным.

**Результат.** shift_failures.json с индексами и предсказаниями на реальных изображениях.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [27]:
# Реализуйте это задание по постановке выше.
shift_failures = []
best_model.eval()

with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(test_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        clean_outputs = best_model(inputs)
        clean_preds = clean_outputs.argmax(dim=1)

        noisy_inputs = add_gaussian_noise(inputs, std=0.3)
        noisy_outputs = best_model(noisy_inputs)
        noisy_preds = noisy_outputs.argmax(dim=1)

        for i in range(len(targets)):
            # Чистый прогноз верный, а зашумленный — ошибочный
            if clean_preds[i] == targets[i] and noisy_preds[i] != targets[i] and len(shift_failures) < 15:
                shift_failures.append({
                    "batch_idx": batch_idx,
                    "sample_idx": i,
                    "true_label": int(targets[i].item()),
                    "clean_prediction": int(clean_preds[i].item()),
                    "noisy_prediction": int(noisy_preds[i].item())
                })
        if len(shift_failures) >= 15:
            break

with open(ARTIFACTS / "shift_failures.json", "w", encoding="utf-8") as f:
    json.dump(shift_failures, f, indent=4, ensure_ascii=False)

print("✔ Анализ сбоев при максимальном сдвиге сохранен в shift_failures.json")

✔ Анализ сбоев при максимальном сдвиге сохранен в shift_failures.json


### Творческий бонус: новый тип сдвига

*Вес: 1 дополнительный балл.*

**Постановка.** Не все повреждения изображений похожи на гауссовский шум.

Выполните следующие действия:
1. Выберите воспроизводимое фотометрическое или геометрическое преобразование test-изображений.
2. Сравните точность с чистым test и приведите три конкретных примера изменения предсказания.

**Результат.** bonus_shift.json с описанием преобразования, метрикой и индексами примеров.

**Критерий принятия.** Задание считается принятым, если описанные действия выполнены, результат соответствует указанному артефакту и получен на предусмотренных данных; в режиме FAST_DEV_RUN результат служит только проверкой исполнения.


In [28]:
# Реализуйте это задание по постановке выше.
import torchvision.transforms.functional as TF

correct_clean, correct_bonus = 0, 0
bonus_failures = []

best_model.eval()
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(test_loader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

        # Творческий бонус: применяем поворот на 45 градусов как новый тип сдвига
        bonus_inputs = torch.stack([TF.rotate(img, angle=45) for img in inputs]).to(DEVICE)

        clean_out = best_model(inputs)
        bonus_out = best_model(bonus_inputs)

        clean_preds = clean_out.argmax(dim=1)
        bonus_preds = bonus_out.argmax(dim=1)

        correct_clean += (clean_preds == targets).sum().item()
        correct_bonus += (bonus_preds == targets).sum().item()

        for i in range(len(targets)):
            if clean_preds[i] == targets[i] and bonus_preds[i] != targets[i] and len(bonus_failures) < 3:
                bonus_failures.append({
                    "true_label": int(targets[i].item()),
                    "clean_pred": int(clean_preds[i].item()),
                    "rotated_pred": int(bonus_preds[i].item())
                })

total_samples = len(test_loader.dataset)
bonus_report = {
    "transformation": "Rotation 45 degrees",
    "clean_accuracy": correct_clean / total_samples,
    "shifted_accuracy": correct_bonus / total_samples,
    "examples": bonus_failures
}

with open(ARTIFACTS / "bonus_shift.json", "w", encoding="utf-8") as f:
    json.dump(bonus_report, f, indent=4, ensure_ascii=False)

print("✔ Творческий бонус выполнен, результаты сохранены в bonus_shift.json")

✔ Творческий бонус выполнен, результаты сохранены в bonus_shift.json


## Анализ результатов

Сопоставьте измеренные показатели на отложенных реальных данных. Укажите бюджет, ошибки модели и ограничения сокращённого запуска.


## Выводы

Сформулируйте предметный вывод по сохранённым артефактам.
